<a href="https://colab.research.google.com/github/Venkata-Prabhath/Crop-Recommendation-Using-Machine-Learning/blob/main/Crop%20Recommendation%20System%20using%20machine%20learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset -
https://soilhealth.dac.gov.in/soilhealthcard

# Load all libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Load Dataset

In [ ]:
data = pd.read_csv("/content/Dataset/Crop_recommendation.csv")

print(data.head())

    N   P   K  temperature   humidity        ph    rainfall label
0  90  42  43    20.879744  82.002744  6.502985  202.935536  rice
1  85  58  41    21.770462  80.319644  7.038096  226.655537  rice
2  60  55  44    23.004459  82.320763  7.840207  263.964248  rice
3  74  35  40    26.491096  80.158363  6.980401  242.864034  rice
4  78  42  42    20.130175  81.604873  7.628473  262.717340  rice


In [ ]:
data.shape

(2200, 8)

In [ ]:
data.describe()

,N,P,K,temperature,humidity,ph,rainfall
count,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000
mean,50.551818,53.362727,48.149091,25.616244,71.481779,6.469480,103.463655
std,36.917334,32.985883,50.647931,5.063749,22.263812,0.773938,54.958389
min,0.000000,5.000000,5.000000,8.825675,14.258040,3.504752,20.211267
25%,21.000000,28.000000,20.000000,22.769375,60.261953,5.971693,64.551686
50%,37.000000,51.000000,32.000000,25.598693,80.473146,6.425045,94.867624
75%,84.250000,68.000000,49.000000,28.561654,89.948771,6.923643,124.267508
max,140.000000,145.000000,205.000000,43.675493,99.981876,9.935091,298.560117


# Set features(x) and target(Y)

In [ ]:
X = data.drop("label", axis=1)
Y = data["label"]

# Train Test Split(80/20)

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)

# Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Evaluation Function

In [ ]:
def evaluate(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average='macro'),
        "Recall": recall_score(y_true, y_pred, average='macro'),
        "F1": f1_score(y_true, y_pred, average='macro')
    }

# Train Models


*   Logistic Regression
*   KNN

*   Decision Tree
*   Random Forest

*   Gradient Boosting
*   XGBoost

*   SVM









#### Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, Y_train)
pred_lr = lr.predict(X_test_scaled)
metrics_lr = evaluate(Y_test, pred_lr)

#### KNN

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, Y_train)
pred_knn = knn.predict(X_test_scaled)
metrics_knn = evaluate(Y_test, pred_knn)

#### Decision Tree

In [ ]:
dt = DecisionTreeClassifier()
dt.fit(X_train_scaled, Y_train)
pred_dt = dt.predict(X_test_scaled)
metrics_dt = evaluate(Y_test, pred_dt)

#### Random Forest

In [ ]:
rf = RandomForestClassifier()
rf.fit(X_train_scaled, Y_train)
pred_rf = rf.predict(X_test_scaled)
metrics_rf = evaluate(Y_test, pred_rf)

#### Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier()
gb.fit(X_train_scaled, Y_train)
pred_gb = gb.predict(X_test_scaled)
metrics_gb = evaluate(Y_test, pred_gb)

#### XGBoost

In [ ]:
le = LabelEncoder()

y_train_enc = le.fit_transform(Y_train)
y_test_enc = le.transform(Y_test)

xgb = XGBClassifier(eval_metric='mlogloss')
xgb.fit(X_train_scaled, y_train_enc)

pred_xgb = xgb.predict(X_test_scaled)
pred_xgb = le.inverse_transform(pred_xgb)

metrics_xgb = evaluate(Y_test, pred_xgb)

#### SVM

In [ ]:
svm = SVC(probability=True)
svm.fit(X_train_scaled, Y_train)
pred_svm = svm.predict(X_test_scaled)
metrics_svm = evaluate(Y_test, pred_svm)

# Compare Models

In [ ]:
results = pd.DataFrame({
    "LR": metrics_lr,
    "KNN": metrics_knn,
    "DT": metrics_dt,
    "RF": metrics_rf,
    "GB": metrics_gb,
    "XGB": metrics_xgb,
    "SVM": metrics_svm
}).T

print(results)

     Accuracy  Precision    Recall        F1
LR   0.972727   0.974022  0.972727  0.972464
KNN  0.979545   0.980356  0.979545  0.979283
DT   0.984091   0.984699  0.984091  0.984128
RF   0.990909   0.991539  0.990909  0.990895
GB   0.988636   0.989742  0.988636  0.988723
XGB  0.993182   0.993506  0.993182  0.993116
SVM  0.984091   0.985610  0.984091  0.984038


# Select Best Model

In [ ]:
best_model = rf

# Top-K Prediction Function

In [ ]:
def top_k_predict(model, X, k=5):
    probs = model.predict_proba(X)
    top_k = np.argsort(probs, axis=1)[:, -k:][:, ::-1]
    labels = model.classes_
    return [[labels[i] for i in row] for row in top_k]

# Example Prediction

In [ ]:
print("Features:\n",X_test.iloc[1],"\n")
print("Actual Crop \n:",Y_test.iloc[1],"\n")
sample = X_test_scaled[1].reshape(1, -1)
print(top_k_predict(best_model, sample, k=5))

Features:
 N              98.000000
P              79.000000
K              50.000000
temperature    25.341198
humidity       84.473213
ph              6.435917
rainfall       91.064934
Name: 1072, dtype: float64 

Actual Crop 
: banana 

[['banana', 'watermelon', 'papaya', 'coffee', 'jute']]


# Best model = XGBoost

In [ ]:
import pickle

pickle.dump(best_model, open("model.pkl", "wb"))
pickle.dump(scaler, open("scaler.pkl", "wb"))
pickle.dump(le, open("encoder.pkl", "wb"))